# fedOPD data fetch

Three datasets, each in its own folder under `../data`:

| # | dataset | folder |
|---|---------|--------|
| 1 | NHS RTT monthly Full CSV zips, Apr 2018 – Mar 2026 (96 files, ~8 GB extracted) | `data/rtt_zips` -> `data/rtt_csv` |
| 2 | Trust -> ICB / geography reference (ODS ORD API) | `data/reference` |
| 3 | KH03 overnight bed availability & occupancy | `data/kh03` |



In [1]:
# ---- libraries -------------------------------------------------------------

import re            # 're' = regular expressions, Python's built-in text-pattern matcher.
                     # NHS filenames look like 'Full-CSV-data-file-Mar19-revised-ZIP-3506K.zip';
                     # we use re to pull just the month token 'Mar19' out of that mess.

import zipfile       # built-in reader for .zip archives. Each RTT month arrives as one zip;
                     # this lets Python check and extract them without any external unzip tool.

from pathlib import Path        # Path objects represent files/folders. Nicer than raw strings:
                                # you join with /, ask .exists(), and the same code works on
                                # Windows and Linux (no \ vs / headaches).

from datetime import datetime   # real calendar dates. We convert 'Mar19' into an actual date
                                # so months sort in time order - sorted as plain text, 'Apr18'
                                # and 'Mar19' would be in alphabetical order, not date order.

import requests      # the standard library for talking to the web: fetch a page, download a
                     # file, call an API. Every byte this notebook grabs comes through requests.

import pandas as pd  # THE data-analysis library. A DataFrame is a table (like a spreadsheet)
                     # you filter, join and save from code. Used here to build the
                     # trust -> ICB reference table and write it out as csv.

from bs4 import BeautifulSoup   # an HTML parser (package 'beautifulsoup4'). A web page arrives
                                # as one giant string of HTML; BeautifulSoup turns it into a
                                # structure you can search: 'give me every link on this page'.

# ---- folder layout: everything lands under repo-root/data ------------------

DATA = Path("../data")                      # '..' = one level up from src/, i.e. repo-root/data
RTT_ZIPS = DATA / "rtt_zips"                # the 96 monthly zips exactly as downloaded (~330 MB)
RTT_CSV = DATA / "rtt_csv"                  # the csvs extracted out of those zips (~8.2 GB)
REF = DATA / "reference"                    # small lookup table: which trust belongs to which ICB
KH03 = DATA / "kh03"                        # quarterly bed-availability spreadsheets

for d in (RTT_ZIPS, RTT_CSV, REF, KH03):    # create all four folders before any download starts
    d.mkdir(parents=True, exist_ok=True)    # parents=True also creates ../data itself if missing;
                                            # exist_ok=True makes rerunning this cell harmless

# NHS web servers reject requests that identify themselves as a Python script
# (HTTP 403 'Forbidden'). Sending a normal browser identity makes them answer.
HEADERS = {"User-Agent": "Mozilla/5.0"}


## 1. Build the RTT Full CSV manifest

Each financial year (2018-19 ... 2025-26) has one page listing that year's monthly files. We scrape all eight pages, keep only links whose filename says Full-CSV and ends in .zip, and key them by the month token in the filename (e.g. `Mar19`). When a month appears twice (original + revised), the revised link wins.

Cross-checked: this filter picks exactly the official links, e.g. `Full-CSV-data-file-Mar19-revised-ZIP-3506K.zip` and `Full-CSV-data-file-Mar26-ZIP-3M-revised.zip`. Ignore the size hint in the filename — the Mar22 revised file says '111805K' but is actually ~4.5 MB on the server; sizes vary wildly month to month and that's fine.

In [2]:
# Goal of this cell: build 'manifest', a dictionary mapping each month ('Mar19')
# to the url of its Full-CSV zip. Nothing is downloaded yet - we are only
# collecting the 96 download links by reading NHS's 8 yearly listing pages.

YEARS = [f"{y}-{str(y + 1)[2:]}" for y in range(2018, 2026)]
# builds the url slugs '2018-19', '2019-20', ... '2025-26'. An f-string pastes
# values into text; str(y + 1)[2:] takes '2019' and keeps everything from
# position 2 onward -> '19'.

manifest = {}                                # will hold: month key -> download url
for yr in YEARS:                             # visit each financial year's page in turn
    page = f"https://www.england.nhs.uk/statistics/statistical-work-areas/rtt-waiting-times/rtt-data-{yr}/"
    r = requests.get(page, headers=HEADERS, timeout=60)   # fetch the page, wait max 60s
    r.raise_for_status()                     # stop loudly if the server said 403/404 -
                                             # otherwise we'd silently collect zero links
    soup = BeautifulSoup(r.text, "html.parser")           # r.text is raw HTML; parse it

    for a in soup.find_all("a", href=True):  # every hyperlink <a href=...> on the page
        href = a["href"]                     # the link's url
        name = href.split("/")[-1]           # text after the last '/' = the filename
        if "full-csv" not in name.lower() or not name.endswith(".zip"):
            continue                         # not the file we want (the page also links
                                             # XLS summaries etc.) -> skip to next link
        key = re.search(r"([A-Z][a-z]{2}\d{2})", name).group(1)
        # the regex pattern means: 1 capital, 2 lowercase letters, 2 digits -> 'Mar19'.
        # .group(1) returns the matched text. If NHS ever changes their filename
        # format there is no match and this line crashes - which is what we want.
        if key not in manifest or "revised" in name.lower():
            manifest[key] = href             # first link wins UNLESS a 'revised' version
                                             # exists - NHS reissues corrected files and
                                             # we always prefer the corrected one

months = sorted(manifest, key=lambda k: datetime.strptime(k, "%b%y"))
# sort the month keys chronologically: strptime('Mar19', '%b%y') converts the
# text to the date 2019-03-01 (%b = month name, %y = 2-digit year), and sorted()
# orders by those dates instead of alphabetically.

assert len(months) == 96, months             # 8 years x 12 months must be 96. If NHS renamed
                                             # links or a page is missing, stop here rather
                                             # than download an incomplete set
print(len(months), "months:", months[0], "->", months[-1])   # expect Apr18 -> Mar26


96 months: Apr18 -> Mar26


## 2. Download all 96 zips

Streamed in 1 MB chunks so big files don't sit in memory. Each file downloads to a `.part` name and is renamed only once fully written — so a completed filename on disk always means a complete download, and rerunning the cell skips finished files and resumes the rest. File sizes vary a lot month to month; that's expected.

In [3]:
# Goal: download each month's zip to disk. Safe to rerun any time - finished
# files are skipped, half-finished ones are redone from scratch.

for k in months:                             # chronological, Apr18 -> Mar26
    url = manifest[k]                        # look up this month's download link
    dest = RTT_ZIPS / url.split("/")[-1]     # final path on disk, keeping the server's filename
    if dest.exists():
        continue                             # fully downloaded on an earlier run -> skip it

    tmp = dest.with_suffix(".part")          # download under a temporary name first,
                                             # e.g. '...-ZIP-3506K.part'

    r = requests.get(url, headers=HEADERS, stream=True, timeout=60)
    # stream=True: don't pull the whole file into memory at once -
    # let the body arrive piece by piece as we read it below
    r.raise_for_status()                     # if the server answered with an error page,
                                             # stop now instead of saving that page as a 'zip'

    with open(tmp, "wb") as f:               # open the temp file for writing bytes ('wb')
        for chunk in r.iter_content(1 << 20):    # read the download 1 MB at a time (1<<20 bytes)
            f.write(chunk)                   # ...appending each piece to the file on disk

    with zipfile.ZipFile(tmp) as z:          # open what we just wrote as a zip archive
        assert z.testzip() is None, tmp.name # testzip() verifies every file inside; None means
                                             # intact. A truncated download fails HERE, while
                                             # still named .part, so it never looks finished

    tmp.rename(dest)                         # only now give it its real name. So a file with
                                             # a final filename is always complete AND valid
    print(k, dest.name, f"{dest.stat().st_size / 1e6:.1f} MB")   # progress line per new file

print("done,", len(list(RTT_ZIPS.glob('*.zip'))), "zips on disk")   # expect 96


Apr18 Full-CSV-data-file-Apr18-revised-ZIP-3380K.zip 3.5 MB
May18 Full-CSV-data-file-May18-revised-ZIP-3424K.zip 3.5 MB
Jun18 Full-CSV-data-file-Jun18-revised-ZIP-3426K.zip 3.5 MB
Jul18 Full-CSV-data-file-Jul18-revised-ZIP-3450K.zip 3.5 MB
Aug18 Full-CSV-data-file-Aug18-revised-ZIP-3464K.zip 3.5 MB
Sep18 Full-CSV-data-file-Sep18-revised-ZIP-K.zip 3.5 MB
Oct18 Full-CSV-data-file-Oct18-revised-ZIP-3574K.zip 3.7 MB
Nov18 Full-CSV-data-file-Nov18-revised-ZIP-3544K.zip 3.6 MB
Dec18 Full-CSV-data-file-Dec18-ZIP-3427K.zip 3.5 MB
Jan19 Full-CSV-data-file-Jan19-revised-ZIP-3634K.zip 3.7 MB
Feb19 Full-CSV-data-file-Feb19-revised-ZIP-3485K.zip 3.6 MB
Mar19 Full-CSV-data-file-Mar19-revised-ZIP-3506K.zip 3.6 MB
Apr19 Full-CSV-data-file-Apr19-ZIP-3436K-revised.zip 3.5 MB
May19 Full-CSV-data-file-May19-ZIP-3497K-revised.zip 3.6 MB
Jun19 Full-CSV-data-file-Jun19-ZIP-3502K.zip 3.6 MB
Jul19 Full-CSV-data-file-Jul19-ZIP-3550K-revised.zip 3.6 MB
Aug19 Full-CSV-data-file-Aug19-ZIP-3493K-revised.zip 3.6 MB


## 3. Extract every zip into one csv folder

In [4]:
# Goal: open every downloaded zip and unpack the csv inside it into one flat
# folder, so later notebooks can simply read data/rtt_csv/*.csv.

for z in sorted(RTT_ZIPS.glob("*.zip")):     # glob('*.zip') lists every zip in the folder
    zipfile.ZipFile(z).extractall(RTT_CSV)   # each zip holds one month's csv; extract it.
                                             # rerunning just overwrites with identical files

print(len(list(RTT_CSV.glob('*.csv'))), "csvs extracted")   # expect 96, ~7.7 GB total


96 csvs extracted


## 4. Trust -> ICB reference (ODS ORD API)

Two steps: one call lists every organisation with primary role RO197 (= NHS Trust), then one call per trust fetches its full record, from which we take the active RE5 ('is commissioned by') relationship — its target is the trust's ICB, which carries primary role **RO261** in ODS. Verified live: e.g. North London NHS FT -> `Z9B2Z` (NHS West and North London ICB).

The list returns ~550 trusts of which ~250 are Active; we keep the inactive ones too since they appear in the 2018-era RTT months (their `icb_code` stays empty). ~550 small requests, a couple of minutes total.

In [5]:
# The RTT csvs identify each hospital trust only by a short code like 'R0A' -
# no name, no region. ODS (Organisation Data Service) is NHS England's registry
# of organisations, and ORD is its REST API: a web service that answers
# structured queries with JSON data instead of an HTML page.
ORD = "https://directory.spineservices.nhs.uk/ORD/2-0-0"

# One request that says: 'list every organisation whose primary role is NHS Trust'.
trusts = requests.get(
    f"{ORD}/organisations",                 # the 'list organisations' endpoint
    params={"PrimaryRoleId": "RO197",       # filter: RO197 is ODS's code for 'NHS Trust',
                                            # which includes both current and closed trusts
            "Limit": 1000,                  # max rows to return; ~550 exist so one call gets all
            "_format": "json"},             # ask for JSON - the default reply is XML
    headers=HEADERS,
    timeout=60,                             # a hung call would otherwise stall the notebook forever
).json()["Organisations"]                   # .json() parses the reply into Python lists/dicts;
                                            # ["Organisations"] unwraps the outer envelope so
                                            # 'trusts' is a plain list, one dict per trust
print(len(trusts), "trusts")                # expect ~544


544 trusts


In [6]:
# Goal: build one table with a row per trust: code, name, status, its ICB (the
# regional NHS body that commissions it) and postcode. The list from the
# previous cell doesn't contain the ICB - that lives inside each trust's full
# record - so we fetch those one at a time (~544 small requests, ~2 minutes).

rows = []                                          # will collect one dict per trust
for t in trusts:
    code = t["OrgId"]                              # e.g. 'R0A' - the SAME code the RTT csvs use,
                                                   # which is what lets us join the tables later
    rec = requests.get(f"{ORD}/organisations/{code}", headers=HEADERS, timeout=60).json()["Organisation"]
    # rec is now the trust's full record: name, address, dates, relationships

    # A record lists relationships to other organisations under 'Rels'. We want
    # the active one of type RE5 ('is commissioned by') whose target is an ICB
    # (role RO261) - that target's code is this trust's ICB.
    icb_code = ""                                  # stays empty for trusts dissolved before ICBs
                                                   # existed (2022) - expected, not an error
    for rel in rec.get("Rels", {}).get("Rel", []): # .get(key, default) instead of [key]: some
                                                   # orgs have no Rels block at all, and .get
                                                   # returns the empty default instead of crashing
        tgt = rel["Target"]                        # the organisation on the other end of the link
        if rel["Status"] == "Active" and rel["id"] == "RE5" and tgt["PrimaryRoleId"]["id"] == "RO261":
            icb_code = tgt["OrgId"]["extension"]   # the ICB's own ODS code, e.g. 'QOP'

    rows.append({                                  # one row of the future table
        "trust_code": code,                        # join key to RTT 'Provider Org Code'
        "trust_name": t["Name"],
        "status": t["Status"],                     # Active / Inactive
        "icb_code": icb_code,                      # empty for dissolved trusts
        "postcode": rec["GeoLoc"]["Location"].get("PostCode", ""),   # for the geography tier
    })

ref = pd.DataFrame(rows)                           # list of dicts -> table, ~544 rows x 5 cols
ref.to_csv(REF / "trust_icb.csv", index=False)     # save it; index=False drops pandas' row numbers
ref.head()                                         # show the first 5 rows as a visual check


,trust_code,trust_name,status,icb_code,postcode
0,G6V2S,NORTH LONDON NHS FOUNDATION TRUST,Active,Z9B2Z,NW1 0PE
1,K0N6A,THE ONLINE NHS TRUST,Active,QKK,SE1 8UG
2,R0A,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,Active,QOP,M13 9WL
3,R0B,SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...,Active,QHM,SR4 7TP
4,R0C,PROJECT NIGHTINGALE NHS TRUST,Inactive,,E16 1XL


## 5. KH03 overnight beds 

The bed-availability page links every quarterly overnight spreadsheet back to 2010 — 61 files as of Jul 2026, latest `Beds-Open-Overnight-Web_File-Q4-2025-26.xlsx` (verified live). We take any xls/xlsx link with 'night' in the filename and download whatever isn't already on disk.

In [7]:
# Goal: download every quarterly 'beds open overnight' spreadsheet. Unlike RTT
# these are small xls/xlsx files linked directly from one page, so no zips.

page = "https://www.england.nhs.uk/statistics/statistical-work-areas/bed-availability-and-occupancy/bed-data-overnight/"
r = requests.get(page, headers=HEADERS, timeout=60)
r.raise_for_status()                        # if NHS blocked the request we'd otherwise
                                            # just 'find' zero links and think we're done
soup = BeautifulSoup(r.text, "html.parser")

# The {...} here builds a SET of urls - a collection where duplicates collapse
# to one entry (pages sometimes link the same file twice). sorted() then turns
# the set into a stable, ordered list.
links = sorted({
    a["href"] for a in soup.find_all("a", href=True)                    # every link on the page
    if a["href"].split("/")[-1].lower().endswith((".xls", ".xlsx"))     # keep spreadsheets only
    and "night" in a["href"].split("/")[-1].lower()                     # 'overnight' beds, not day-case
})
print(len(links), "files")                  # expect 61, back to 2011

for url in links:
    dest = KH03 / url.split("/")[-1]        # save under the server's own filename
    if dest.exists():
        continue                            # already on disk from an earlier run -> skip
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()                    # an html error page saved as '.xlsx' would
                                            # break the spreadsheet reader later
    dest.write_bytes(r.content)             # files are a few hundred KB, so write in one
                                            # go - no streaming needed at this size
    print(dest.name)                        # progress line per new file


61 files
Beds-Open-Overnight-Web_File-Final-Q1-2015-16-52341.xlsx
Beds-Open-Overnight-Web_File-Final-Q2-2015-16-38498.xlsx
Beds-Open-Overnight-Web_File-Final-Q3-2015-16-39849.xlsx
Beds-Open-Overnight-Web_File-Q1-2011-12-November-2012-Refresh-v3-Final.xls
Beds-Open-Overnight-Web_File-Q1-2012-13-May-2013-Refresh-Final-47384.xls
Beds-Open-Overnight-Web_File-Q1-2013-14-Nov-Revised-Revised-Final.xls
Beds-Open-Overnight-Web_File-Q1-2014-15-Revised-Nov15-Final-21447.xlsx
Beds-Open-Overnight-Web_File-Q2-2011-12-November-2012-Refresh-v2-Final.xls
Beds-Open-Overnight-Web_File-Q2-2012-13-May-2013-Refresh-Final-8947-v2.xls
Beds-Open-Overnight-Web_File-Q2-2013-14-Revised-Final1.xls
Beds-Open-Overnight-Web_File-Q2-2014-15-Revised-Nov15-Final-41447.xlsx
Beds-Open-Overnight-Web_File-Q3-2011-12-November-2012-Refresh-v2-Final.xls
Beds-Open-Overnight-Web_File-Q3-2012-13-Revised-Final-76557.xls
Beds-Open-Overnight-Web_File-Q3-2013-14-Revised-Final1.xls
Beds-Open-Overnight-Web_File-Q3-2014-15-Revised-Nov15

## 6. Sanity check

One glance answers "did the fetch complete": four counts, read off the disk, not from memory.

In [8]:
# Counts read directly off the DISK, not from variables in memory - so this
# cell answers 'did the fetch actually complete?' even in a fresh session
# (except ref.shape, which needs the section-4 cells run first).

print("rtt zips :", len(list(RTT_ZIPS.glob("*.zip"))))   # expect 96
print("rtt csvs :", len(list(RTT_CSV.glob("*.csv"))))    # expect 96
print("reference:", ref.shape)                           # expect ~(544, 5)
print("kh03     :", len(list(KH03.glob("*.xls*"))))      # expect 61


rtt zips : 96
rtt csvs : 96
reference: (544, 5)
kh03     : 61
